# Spam Mail Prediction using Machine Learning
**Project by:** Arjun | Internship Project

---
### Table of Contents
1. Importing the Dependencies
2. Data Collection & Pre-Processing
3. **Phase 1: Exploratory Data Analysis (EDA)**
4. Label Encoding
5. Feature Extraction (TF-IDF)
6. Model Training (Logistic Regression)
7. Model Evaluation
8. Building a Predictive System

## 1. Importing the Dependencies

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# --- Phase 1: EDA libraries ---
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
import string
import warnings
warnings.filterwarnings('ignore')

# Set global plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)

print("All libraries imported successfully!")

## 2. Data Collection & Pre-Processing

In [ ]:
# Loading the data from csv file to a pandas Dataframe
# Dataset path: upload the 'data' folder to /content/ in Google Colab
raw_mail_data = pd.read_csv('/content/data/spam.csv', encoding='latin-1')

# Keep only the first two relevant columns and rename them
raw_mail_data = raw_mail_data[['v1', 'v2']]
raw_mail_data.columns = ['Category', 'Message']

print("Dataset loaded successfully!")
print(f"Shape: {raw_mail_data.shape}")

In [ ]:
print(raw_mail_data)

In [ ]:
# Replace the null values with an empty string
mail_data = raw_mail_data.where((pd.notnull(raw_mail_data)), '')

# Printing the first 5 rows of the dataframe
mail_data.head()

In [ ]:
# Checking the number of rows and columns in the dataframe
print("Shape:", mail_data.shape)
print("\nData Types:")
print(mail_data.dtypes)
print("\nNull values:")
print(mail_data.isnull().sum())

---
## 3. Phase 1: Exploratory Data Analysis (EDA)

> In this phase we deeply understand the dataset before building any model.
> We answer questions like: How balanced is the data? How long are spam messages vs ham? What words appear most in spam?

---

### 3.1 Class Distribution (Spam vs Ham)

In [ ]:
# Count of spam vs ham messages
class_counts = mail_data['Category'].value_counts()
print("Class Distribution:")
print(class_counts)
print(f"\nSpam percentage  : {class_counts['spam'] / len(mail_data) * 100:.2f}%")
print(f"Ham  percentage  : {class_counts['ham']  / len(mail_data) * 100:.2f}%")

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
colors = ['#e74c3c', '#2ecc71']
axes[0].bar(class_counts.index, class_counts.values, color=colors, edgecolor='black', width=0.5)
axes[0].set_title('Spam vs Ham — Count', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Category', fontsize=12)
axes[0].set_ylabel('Number of Messages', fontsize=12)
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold', fontsize=12)

# Pie chart
axes[1].pie(
    class_counts.values,
    labels=class_counts.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=140,
    explode=(0.05, 0),
    shadow=True,
    textprops={'fontsize': 12}
)
axes[1].set_title('Spam vs Ham — Proportion', fontsize=14, fontweight='bold')

plt.suptitle('Class Distribution in SMS Spam Dataset', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\nObservation: The dataset is IMBALANCED — ~87% ham and ~13% spam.")
print("This means accuracy alone is not a reliable metric. We will address this in Phase 6.")

### 3.2 Message Length Analysis

In [ ]:
# Engineer length-based features for EDA
mail_data['char_count']  = mail_data['Message'].apply(len)
mail_data['word_count']  = mail_data['Message'].apply(lambda x: len(x.split()))
mail_data['sent_count']  = mail_data['Message'].apply(lambda x: x.count('.') + x.count('!') + x.count('?'))

# Summary statistics per class
print("=" * 60)
print("Summary Statistics: Character Count")
print("=" * 60)
print(mail_data.groupby('Category')['char_count'].describe().round(2))

print("\n" + "=" * 60)
print("Summary Statistics: Word Count")
print("=" * 60)
print(mail_data.groupby('Category')['word_count'].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

spam_data = mail_data[mail_data['Category'] == 'spam']
ham_data  = mail_data[mail_data['Category'] == 'ham']

# --- Character count histogram ---
axes[0].hist(spam_data['char_count'], bins=50, alpha=0.7, color='#e74c3c', label='Spam', edgecolor='black')
axes[0].hist(ham_data['char_count'],  bins=50, alpha=0.7, color='#2ecc71', label='Ham',  edgecolor='black')
axes[0].axvline(spam_data['char_count'].mean(), color='red',   linestyle='--', linewidth=1.5, label=f'Spam mean: {spam_data["char_count"].mean():.0f}')
axes[0].axvline(ham_data['char_count'].mean(),  color='green', linestyle='--', linewidth=1.5, label=f'Ham mean:  {ham_data["char_count"].mean():.0f}')
axes[0].set_title('Character Count Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Characters', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].legend(fontsize=10)

# --- Word count histogram ---
axes[1].hist(spam_data['word_count'], bins=40, alpha=0.7, color='#e74c3c', label='Spam', edgecolor='black')
axes[1].hist(ham_data['word_count'],  bins=40, alpha=0.7, color='#2ecc71', label='Ham',  edgecolor='black')
axes[1].axvline(spam_data['word_count'].mean(), color='red',   linestyle='--', linewidth=1.5, label=f'Spam mean: {spam_data["word_count"].mean():.0f}')
axes[1].axvline(ham_data['word_count'].mean(),  color='green', linestyle='--', linewidth=1.5, label=f'Ham mean:  {ham_data["word_count"].mean():.0f}')
axes[1].set_title('Word Count Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Words', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].legend(fontsize=10)

plt.suptitle('Message Length: Spam vs Ham', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("Observation: Spam messages tend to be LONGER than ham messages.")
print(f"  Spam avg char count: {spam_data['char_count'].mean():.1f}")
print(f"  Ham  avg char count: {ham_data['char_count'].mean():.1f}")

### 3.3 Box Plot — Length Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Character count boxplot
sns.boxplot(
    data=mail_data, x='Category', y='char_count',
    palette={'spam': '#e74c3c', 'ham': '#2ecc71'},
    ax=axes[0], width=0.5
)
axes[0].set_title('Character Count per Class', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Category', fontsize=11)
axes[0].set_ylabel('Character Count', fontsize=11)

# Word count boxplot
sns.boxplot(
    data=mail_data, x='Category', y='word_count',
    palette={'spam': '#e74c3c', 'ham': '#2ecc71'},
    ax=axes[1], width=0.5
)
axes[1].set_title('Word Count per Class', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Category', fontsize=11)
axes[1].set_ylabel('Word Count', fontsize=11)

plt.suptitle('Box Plot — Spam vs Ham Message Lengths', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 3.4 WordCloud — Most Frequent Words

In [ ]:
# Install wordcloud if not already available
# !pip install wordcloud   # Uncomment this line if running for the first time on Colab

In [ ]:
# Combine all spam messages into one text blob
spam_text = ' '.join(spam_data['Message'].values)
ham_text  = ' '.join(ham_data['Message'].values)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Spam WordCloud ---
spam_wc = WordCloud(
    width=800, height=400,
    background_color='black',
    colormap='Reds',
    max_words=150,
    collocations=False,
    stopwords=None
).generate(spam_text)

axes[0].imshow(spam_wc, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('🚨 SPAM Messages — Top Words', fontsize=14, fontweight='bold', color='#e74c3c')

# --- Ham WordCloud ---
ham_wc = WordCloud(
    width=800, height=400,
    background_color='black',
    colormap='Greens',
    max_words=150,
    collocations=False,
    stopwords=None
).generate(ham_text)

axes[1].imshow(ham_wc, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('✅ HAM Messages — Top Words', fontsize=14, fontweight='bold', color='#2ecc71')

plt.suptitle('WordCloud: Most Frequent Words in Spam vs Ham', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("Observation: Spam messages heavily feature words like 'FREE', 'call', 'win', 'prize', 'text', 'claim'.")
print("Ham messages are more conversational — 'ok', 'come', 'get', 'going', 'know', 'like'.")

### 3.5 Top 20 Most Frequent Words — Spam vs Ham

In [ ]:
import re

def get_top_words(text_series, n=20):
    """Extract top-N most frequent words from a Series of text messages."""
    all_words = []
    for msg in text_series:
        # Lowercase, remove punctuation, split
        words = re.sub(r'[^\w\s]', '', msg.lower()).split()
        all_words.extend(words)
    return Counter(all_words).most_common(n)

top_spam = get_top_words(spam_data['Message'], n=20)
top_ham  = get_top_words(ham_data['Message'],  n=20)

spam_words, spam_freq = zip(*top_spam)
ham_words,  ham_freq  = zip(*top_ham)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Spam top words ---
axes[0].barh(list(spam_words)[::-1], list(spam_freq)[::-1],
             color='#e74c3c', edgecolor='black', alpha=0.85)
axes[0].set_title('🚨 Top 20 Words in SPAM Messages', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Frequency', fontsize=11)
for i, (w, f) in enumerate(zip(reversed(spam_words), reversed(spam_freq))):
    axes[0].text(f + 5, i, str(f), va='center', fontsize=9)

# --- Ham top words ---
axes[1].barh(list(ham_words)[::-1], list(ham_freq)[::-1],
             color='#2ecc71', edgecolor='black', alpha=0.85)
axes[1].set_title('✅ Top 20 Words in HAM Messages', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Frequency', fontsize=11)
for i, (w, f) in enumerate(zip(reversed(ham_words), reversed(ham_freq))):
    axes[1].text(f + 5, i, str(f), va='center', fontsize=9)

plt.suptitle('Top 20 Most Frequent Words: Spam vs Ham', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.6 Character-Level Feature Analysis

In [ ]:
# Engineer character-level features
mail_data['uppercase_ratio']   = mail_data['Message'].apply(
    lambda x: sum(1 for c in x if c.isupper()) / (len(x) + 1)
)
mail_data['digit_count']       = mail_data['Message'].apply(
    lambda x: sum(1 for c in x if c.isdigit())
)
mail_data['exclamation_count'] = mail_data['Message'].apply(
    lambda x: x.count('!')
)
mail_data['has_url']           = mail_data['Message'].apply(
    lambda x: 1 if re.search(r'http|www|bit.ly|\.com', x, re.IGNORECASE) else 0
)
mail_data['has_currency']      = mail_data['Message'].apply(
    lambda x: 1 if re.search(r'[\$£€]|free|win|prize|cash|reward', x, re.IGNORECASE) else 0
)

# Summary comparison table
features = ['char_count', 'word_count', 'uppercase_ratio', 'digit_count',
            'exclamation_count', 'has_url', 'has_currency']

summary = mail_data.groupby('Category')[features].mean().round(3).T
summary.columns.name = None
summary.index.name = 'Feature'

print("=" * 55)
print("Feature Averages by Class")
print("=" * 55)
print(summary.to_string())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
plot_features = [
    ('uppercase_ratio',   'Uppercase Letter Ratio'),
    ('digit_count',       'Digit Count'),
    ('exclamation_count', 'Exclamation Mark Count'),
    ('char_count',        'Character Count'),
    ('word_count',        'Word Count'),
    ('has_url',           'Has URL (0/1)')
]

palette = {'spam': '#e74c3c', 'ham': '#2ecc71'}

for ax, (feat, label) in zip(axes.flatten(), plot_features):
    spam_val = mail_data[mail_data['Category'] == 'spam'][feat].mean()
    ham_val  = mail_data[mail_data['Category'] == 'ham'][feat].mean()
    bars = ax.bar(['Spam', 'Ham'], [spam_val, ham_val],
                  color=['#e74c3c', '#2ecc71'], edgecolor='black', width=0.4)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_ylabel('Average Value', fontsize=9)
    for bar, val in zip(bars, [spam_val, ham_val]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Character-Level Feature Comparison: Spam vs Ham', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("  - Spam messages have a HIGHER uppercase ratio → agressive marketing tone")
print("  - Spam messages have MORE digits → phone numbers, prize amounts")
print("  - Spam messages have MORE exclamation marks → urgency")
print("  - Spam messages more frequently contain URLs")

### 3.7 Correlation Heatmap of Engineered Features

In [ ]:
# Create a numeric version of Category for correlation (spam=0, ham=1)
mail_data['label'] = mail_data['Category'].map({'spam': 0, 'ham': 1})

corr_features = ['label', 'char_count', 'word_count', 'uppercase_ratio',
                 'digit_count', 'exclamation_count', 'has_url', 'has_currency']

corr_matrix = mail_data[corr_features].corr()

plt.figure(figsize=(10, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f',
    cmap='RdYlGn',
    mask=mask,
    linewidths=0.5,
    vmin=-1, vmax=1,
    square=True,
    cbar_kws={'shrink': 0.8}
)
plt.title('Correlation Heatmap — Engineered Features vs Label', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print top correlations with label
print("\nCorrelation with Label (spam=0, ham=1) — sorted:")
label_corr = corr_matrix['label'].drop('label').sort_values()
for feat, val in label_corr.items():
    direction = "(spam indicator ↑)" if val < 0 else "(ham indicator ↑)"
    print(f"  {feat:<22}: {val:+.3f}  {direction}")

### 3.8 EDA Summary

| Finding | Detail |
|---|---|
| **Dataset size** | 5,572 messages |
| **Class imbalance** | ~87% ham, ~13% spam — imbalanced! |
| **Spam messages are longer** | Avg ~139 chars vs ~72 chars for ham |
| **Spam uses more uppercase** | Aggressive marketing tone |
| **Spam uses more digits** | Phone numbers, prize amounts |
| **Spam uses more exclamations** | Urgency-driven language |
| **Key spam words** | FREE, call, win, prize, text, claim, urgent |
| **Key ham words** | ok, come, get, going, know, like, time |

> **Next Steps:** In Phase 2, we will use these insights to build better NLP features using NLTK (stemming, lemmatization) and add the engineered features to improve model performance.

---
## 4. Label Encoding

In [ ]:
# Label spam mail as 0; ham mail as 1
mail_data.loc[mail_data['Category'] == 'spam', 'Category'] = 0
mail_data.loc[mail_data['Category'] == 'ham',  'Category'] = 1

print("Label encoding complete:")
print("  spam  →  0")
print("  ham   →  1")
print(mail_data['Category'].value_counts())

In [ ]:
# Separating the data as texts and labels
X = mail_data['Message']
Y = mail_data['Category']

print("X shape:", X.shape)
print("Y shape:", Y.shape)

---
## 5. Splitting the Data into Training & Test Sets

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=3)

print("Split Summary:")
print(f"  Total  : {X.shape[0]}")
print(f"  Train  : {X_train.shape[0]} ({X_train.shape[0]/X.shape[0]*100:.1f}%)")
print(f"  Test   : {X_test.shape[0]}  ({X_test.shape[0]/X.shape[0]*100:.1f}%)")

---
## 6. Feature Extraction (TF-IDF)

In [ ]:
# Transform the text data to feature vectors using TF-IDF
feature_extraction = TfidfVectorizer(min_df=1, stop_words='english', lowercase=True)

X_train_features = feature_extraction.fit_transform(X_train)
X_test_features  = feature_extraction.transform(X_test)

# Convert Y_train and Y_test values to integers
Y_train = Y_train.astype('int')
Y_test  = Y_test.astype('int')

print("TF-IDF Feature Matrix:")
print(f"  Training features shape : {X_train_features.shape}")
print(f"  Test features shape     : {X_test_features.shape}")
print(f"  Vocabulary size         : {len(feature_extraction.vocabulary_)}")

---
## 7. Training the Model (Logistic Regression)

In [ ]:
# Training the Logistic Regression model with the training data
model = LogisticRegression(max_iter=1000)
model.fit(X_train_features, Y_train)

print("Model training complete!")

---
## 8. Evaluating the Trained Model

In [ ]:
# Prediction on training data
prediction_on_training_data = model.predict(X_train_features)
accuracy_on_training_data   = accuracy_score(Y_train, prediction_on_training_data)

# Prediction on test data
prediction_on_test_data = model.predict(X_test_features)
accuracy_on_test_data   = accuracy_score(Y_test, prediction_on_test_data)

print("=" * 40)
print("Model Evaluation (Accuracy Score)")
print("=" * 40)
print(f"  Training Accuracy : {accuracy_on_training_data * 100:.2f}%")
print(f"  Test     Accuracy : {accuracy_on_test_data * 100:.2f}%")
print("=" * 40)
print("\nNote: In Phase 4, we will add Confusion Matrix, ROC Curve,")
print("Precision-Recall, and F1-Score for a more complete evaluation.")

---
## 9. Building a Predictive System

In [ ]:
def predict_mail(message):
    """Classify a single email/SMS message as Spam or Ham."""
    input_data_features = feature_extraction.transform([message])
    prediction = model.predict(input_data_features)
    probability = model.predict_proba(input_data_features)[0]

    label = 'HAM (Legitimate)' if prediction[0] == 1 else 'SPAM'
    confidence = probability[prediction[0]] * 100

    print(f"Message   : {message[:80]}..." if len(message) > 80 else f"Message   : {message}")
    print(f"Prediction: {label}")
    print(f"Confidence: {confidence:.2f}%")
    print("-" * 50)

# --- Test with example messages ---
test_messages = [
    "I've been searching for the right words to thank you for this breather. I promise i wont take your help for granted and will fulfil my promise. You have been wonderful and a blessing at all times",
    "FREE entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's",
    "Hey, are you coming to the party tonight? Let me know!",
    "Congratulations! You've won a £1000 Walmart gift card. Click here to claim: www.win-prizes.com",
    "Mom says dinner is ready, come home now"
]

print("=" * 50)
print("PREDICTIVE SYSTEM — Test Results")
print("=" * 50)
for msg in test_messages:
    predict_mail(msg)